# SPIDER — Notebook 5: dataset figures for the manuscript

Three figures, saved to `MyDrive/spider/figures/` at 300 dpi:

| file | content |
|---|---|
| `fig_dataset.png` | mid-sagittal T2 from three scanners, with reference masks overlaid |
| `fig_grades.png` | disc crops illustrating each grade (Pfirrmann 1-5, narrowing, Modic) |
| `fig_real_measurement.png` | the extractor's frame and body isolation drawn on a real disc |

**No GPU needed** — a normal CPU runtime is fine. Run the cells in order.
The images download once (about 3.7 GB, 5-15 minutes).

SPIDER is CC-BY 4.0, so these images may be published with attribution; the
captions in the manuscript credit the dataset.

In [ ]:
# ---------- Cell 1: setup and data ----------
import os, glob, re, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/spider'
FIG = f'{DRIVE}/figures'; os.makedirs(FIG, exist_ok=True)

if not glob.glob('spider_masks/**/*.mha', recursive=True):
    !wget -q -O masks.zip "https://zenodo.org/records/10159290/files/masks.zip?download=1"
    !unzip -q -o masks.zip -d spider_masks
if not glob.glob('spider_images/**/*.mha', recursive=True):
    !wget -q --show-progress -O images.zip "https://zenodo.org/records/10159290/files/images.zip?download=1"
    !unzip -q -o images.zip -d spider_images
if not os.path.exists('overview.csv'):
    !wget -q -O overview.csv "https://zenodo.org/records/10159290/files/overview.csv?download=1"
!pip -q install SimpleITK
import SimpleITK as sitk

def index(root):
    out = {}
    for p in glob.glob(f'{root}/**/*.mha', recursive=True):
        m = re.match(r'(\d+)_(.+)', os.path.basename(p)[:-4])
        if m: out[(int(m.group(1)), m.group(2))] = p
    return out
IMG, MSK = index('spider_images'), index('spider_masks')
print(f"images {len(IMG)}, masks {len(MSK)}, paired {len(set(IMG) & set(MSK))}")

In [ ]:
# ---------- Cell 2: shared helpers -- orient a sagittal slice consistently ----------
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
VERT_MAX, CANAL, DISC = 99, 100, 200

def load(pat, seq):
    im, mk = sitk.ReadImage(IMG[(pat, seq)]), sitk.ReadImage(MSK[(pat, seq)])
    ia = sitk.GetArrayFromImage(im).astype(np.float32)
    ma = sitk.GetArrayFromImage(mk).astype(np.int16)
    sx, sy, sz = im.GetSpacing()
    return ia, ma, np.array([sz, sy, sx])

def axes_of(ma):
    # (si, ap, lr, superior_sign, posterior_sign) derived from the mask, as in Notebook 1
    lr = int(np.argmin(ma.shape)); rest = [a for a in range(3) if a != lr]
    labs = [l for l in np.unique(ma) if 1 <= l <= VERT_MAX]
    cent = {l: np.argwhere(ma == l).mean(0) for l in labs}
    best, br, slope = rest[0], -1, 1
    for a in rest:
        v = np.array([cent[l][a] for l in labs])
        if v.std() == 0: continue
        r = abs(np.corrcoef(labs, v)[0, 1])
        if r > br: best, br, slope = a, r, np.polyfit(labs, v, 1)[0]
    ap = [a for a in rest if a != best][0]
    allv = np.vstack([np.argwhere(ma == l) for l in labs])
    post = np.sign(np.argwhere(ma == CANAL)[:, ap].mean() - allv[:, ap].mean()) if (ma == CANAL).any() else 1
    return best, ap, lr, np.sign(slope), post

def midsagittal(ia, ma, sp):
    # slice through the canal midline, returned as (SI, AP) with superior up and posterior right
    si, ap, lr, sup, post = axes_of(ma)
    s = int(np.argmax((ma == CANAL).sum(axis=tuple(a for a in range(3) if a != lr))))
    img, msk = np.take(ia, s, axis=lr), np.take(ma, s, axis=lr)
    order = [a for a in range(3) if a != lr]                  # remaining axes, in order
    if order.index(si) != 0: img, msk = img.T, msk.T         # rows = SI
    if sup > 0: img, msk = img[::-1], msk[::-1]              # superior at the top
    if post < 0: img, msk = img[:, ::-1], msk[:, ::-1]       # posterior on the right
    ext = [0, img.shape[1] * sp[ap], img.shape[0] * sp[si], 0]
    return img, msk, ext

def window(img):
    lo, hi = np.percentile(img, [1, 99.5]); return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
print("helpers ready")

In [ ]:
# ---------- Cell 3: Fig. dataset -- annotated mid-sagittal T2 from three scanners ----------
ov = pd.read_csv('overview.csv')
ov['Patient'] = ov['new_file_name'].str.split('_').str[0].astype(int)
ov['seq'] = ov['new_file_name'].str.split('_', n=1).str[1]
PICK = ['Ingenia', 'Avanto_fit', 'SymphonyTim']
chosen = []
for sc in PICK:
    c = ov[(ov.ManufacturerModelName == sc) & (ov.seq == 't2')]
    c = [p for p in c.Patient if (p, 't2') in IMG and (p, 't2') in MSK]
    if c: chosen.append((sc, sorted(c)[len(c) // 2]))        # a mid-list patient, not the first
print("patients shown:", chosen)

lab_cmap = ListedColormap([(0,0,0,0), (0.38,0.62,0.85,.45), (0.99,0.55,0.24,.65), (0.30,0.75,0.40,.45)])
fig, axs = plt.subplots(1, 2 * len(chosen), figsize=(3.1 * 2 * len(chosen) / 1.25, 5.2))
for k, (sc, pat) in enumerate(chosen):
    ia, ma, sp = load(pat, 't2'); img, msk, ext = midsagittal(ia, ma, sp)
    ov_lab = np.zeros_like(msk, np.int8)
    ov_lab[(msk >= 1) & (msk <= VERT_MAX)] = 1; ov_lab[msk > DISC] = 2; ov_lab[msk == CANAL] = 3
    a, b = axs[2 * k], axs[2 * k + 1]
    a.imshow(window(img), cmap='gray', extent=ext); a.set_title(f'{sc}: T2', fontsize=9)
    b.imshow(window(img), cmap='gray', extent=ext)
    b.imshow(ov_lab, cmap=lab_cmap, vmin=0, vmax=3, extent=ext, interpolation='nearest')
    ry, rx = ext[2] / msk.shape[0], ext[1] / msk.shape[1]
    for d in [l for l in np.unique(msk) if DISC < l < DISC + 6]:
        rr, cc = np.nonzero(msk == d)
        b.text(cc.min() * rx - 4, rr.mean() * ry, str(d - DISC),
               color='yellow', fontsize=8, ha='right', va='center', weight='bold')
    b.set_title('reference masks', fontsize=9)
    for x in (a, b): x.set_xticks([]); x.set_yticks([])
fig.legend(handles=[Patch(fc=(0.38,0.62,0.85),label='vertebrae'), Patch(fc=(0.99,0.55,0.24),label='intervertebral discs'),
                    Patch(fc=(0.30,0.75,0.40),label='spinal canal')], loc='lower center', ncol=3, fontsize=8, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.savefig(f'{FIG}/fig_dataset.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ---------- Cell 4: Fig. grades -- example crops for each grade ----------
prim = pd.read_csv(f'{DRIVE}/spider_primary_cohort.csv')
z = np.load(f'{DRIVE}/spider_crops_tight.npz', allow_pickle=True)
X, ri = z['images'], z['row_index']
prim = prim.loc[ri].reset_index(drop=True)
rng = np.random.default_rng(3)
def pick(mask):
    idx = np.flatnonzero(mask.values); return int(rng.choice(idx)) if len(idx) else None

panels = [(f'Pfirrmann {g}', pick(prim.pfirrmann == g)) for g in range(1, 6)]
panels += [('narrowing: no', pick((prim.narrowing == 0) & (prim.pfirrmann <= 3))),
           ('narrowing: yes', pick((prim.narrowing == 1) & (prim.pfirrmann >= 3))),
           ('Modic: none', pick(prim.modic_any == 0)),
           ('Modic: present', pick(prim.modic_any == 1))]
fig, axs = plt.subplots(2, 5, figsize=(12.5, 5.4))
for ax, (title, i) in zip(axs.ravel(), panels):
    ax.imshow(X[i], cmap='gray'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{title}\nlevel {int(prim.ivd_label[i])}', fontsize=9)
axs.ravel()[-1].axis('off')
plt.tight_layout(); plt.savefig(f'{FIG}/fig_grades.png', dpi=300, bbox_inches='tight'); plt.show()
print("indices used:", [i for _, i in panels])

In [ ]:
# ---------- Cell 5: Fig. real measurement -- frame and body isolation on a real disc ----------
pat, seq, level = chosen[0][1], 't2', 3             # patient from the largest scanner, disc level 3
ia, ma, sp = load(pat, seq); img, msk, ext = midsagittal(ia, ma, sp)
ry, rx = ext[2] / msk.shape[0], ext[1] / msk.shape[1]  # mm per row, mm per column
d = DISC + level
dr, dc = np.nonzero(msk == d)
P = np.c_[dr * ry, dc * rx]                            # disc pixels in mm (SI, AP)
c = P.mean(0); _, _, V = np.linalg.svd(P - c, full_matrices=False); n = V[1]
if n[0] > 0: n = -n                                    # point superiorly (rows grow downward)
t = (P - c) @ n; h = np.percentile(t, 99) - np.percentile(t, 1)
lo, hi = np.percentile(P[:, 1], [2, 98])               # disc AP footprint

# neighbouring vertebrae found geometrically: nearest centroid above and below the disc
vcent = {v: np.nonzero(msk == v)[0].mean() * ry for v in np.unique(msk) if 1 <= v <= VERT_MAX}
above = [v for v, r in vcent.items() if r < c[0]]; below = [v for v, r in vcent.items() if r > c[0]]
neigh = [max(above, key=lambda v: vcent[v]) if above else None, min(below, key=lambda v: vcent[v]) if below else None]
lab = np.zeros_like(msk, np.int8)
for v in [x for x in neigh if x is not None]:
    rr, cc = np.nonzero(msk == v); inside = (cc * rx >= lo) & (cc * rx <= hi)
    lab[rr[inside], cc[inside]] = 1; lab[rr[~inside], cc[~inside]] = 2
lab[msk == d] = 3; lab[msk == CANAL] = 4
cm = ListedColormap([(0,0,0,0), (0.38,0.62,0.85,.55), (0.75,0.75,0.75,.6), (0.99,0.55,0.24,.7), (0.30,0.75,0.40,.5)])

fig, axs = plt.subplots(1, 2, figsize=(10, 5.2))
for ax, over in zip(axs, (False, True)):
    ax.imshow(window(img), cmap='gray', extent=ext)
    if over:
        ax.imshow(lab, cmap=cm, vmin=0, vmax=4, extent=ext, interpolation='nearest')
        ax.annotate('', xy=(c[1] + 14 * n[1], c[0] + 14 * n[0]), xytext=(c[1], c[0]),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.6))
        ax.annotate('', xy=(c[1] + h / 2 * n[1], c[0] + h / 2 * n[0]), xytext=(c[1] - h / 2 * n[1], c[0] - h / 2 * n[0]),
                    arrowprops=dict(arrowstyle='<->', color='yellow', lw=1.3))
        ax.text(c[1] - 22, c[0], f'{h:.1f} mm', color='yellow', fontsize=9, ha='right', va='center', weight='bold')
        ax.legend(handles=[Patch(fc=(0.38,0.62,0.85),label='vertebral body (measured)'),
                           Patch(fc=(0.75,0.75,0.75),label='posterior arch (excluded)'),
                           Patch(fc=(0.99,0.55,0.24),label=f'disc, level {level}'),
                           Patch(fc=(0.30,0.75,0.40),label='spinal canal')], fontsize=7, loc='lower left')
    ax.set_xlim(c[1] - 45, c[1] + 55); ax.set_ylim(c[0] + 45, c[0] - 45)
    ax.set_xticks([]); ax.set_yticks([])
axs[0].set_title(f'patient {pat}, T2, level {level}', fontsize=9)
axs[1].set_title('disc normal (red), thickness (yellow), body vs arch', fontsize=9)
plt.tight_layout(); plt.savefig(f'{FIG}/fig_real_measurement.png', dpi=300, bbox_inches='tight'); plt.show()
print(f"patient {pat}, level {level}: in-plane thickness {h:.1f} mm")
print(f"\nall three figures saved in {FIG} -- download them and send them to me")